# Two-Dimensional Vortex Panel Method
## Results, Discussion and Selected Implementation Details
 
**Student name:** [ANBUCHERAN S]  
**Student ID:** [SC26M005]  

---

## Abstract

This report presents the results and discussion of constant-strength and linear-strength vortex panel methods for two-dimensional potential flow. The methods are verified using a circular cylinder and applied to symmetric NACA 0012 and cambered NACA 4412 airfoils. The main investigations are panel-number convergence, panel-spacing effects, surface pressure-coefficient distributions, and the influence of airfoil thickness.

The emphasis is on numerical results and their physical interpretation. Only selected implementation snippets are included; the complete program is maintained separately in the working notebook. Numerical values below are transcribed from the saved outputs of the supplied `Linear_VPM.ipynb` notebook.

## 1. Numerical model and selected implementation details

The body contour is discretised into straight panels. The unknown vortex-sheet strengths are determined by enforcing the no-penetration condition at panel control points. The tangential velocity is then used to calculate the pressure coefficient:

$$
C_p=1-\left(\frac{V_t}{V_\infty}\right)^2.
$$

The airfoil calculations use a trailing-edge Kutta condition. In the constant-strength implementation the first and last panel strengths are constrained to be equal and opposite. In the linear-strength implementation the first and last nodal strengths are constrained to be equal and opposite.

### 1.1 Main functions used

| Function | Purpose |
|---|---|
| `define_panels(x, y)` | Creates panel geometry, control points, lengths, tangents and normals |
| `calculate_influence_integrals(...)` | Evaluates analytical influence integrals for linear-strength panels |
| `calculate_velocity_coefficients(panels)` | Builds normal and tangential influence matrices for the linear method |
| `assemble_linear_system(...)` | Forms the linear-strength system including the Kutta condition |
| `solve_vortex_strengths(M, b)` | Solves the linear algebraic system |
| `tangential_velocity_and_cp(...)` | Computes surface tangential velocity and pressure coefficient |
| `calculate_constant_velocity_coefficients(panels)` | Builds the constant-strength influence matrices |
| `assemble_constant_airfoil_system(...)` | Forms the constant-strength airfoil system |
| `vortex_panel_method_0(...)` | Wrapper for the constant-strength airfoil method |
| `vortex_panel_method_1(...)` | Wrapper for the linear-strength airfoil method |
| `generate_naca_0012(...)`, `generate_naca_4412(...)` | Generate symmetric and cambered airfoil geometries |
| `generate_naca00xx(...)` | Generates symmetric airfoils for the thickness study |

### 1.2 Required method interfaces

The assignment requested two callable functions with arguments and returned data. The implemented wrappers are called as follows:

```python
result_0 = vortex_panel_method_0(panels, V_inf, alpha)  # constant strength
result_1 = vortex_panel_method_1(panels, V_inf, alpha)  # linear strength
```

Both return a dictionary containing the vortex strengths (`gamma`), tangential velocity (`V_t`), pressure coefficient (`Cp`), influence matrices, assembled system matrix and right-hand side, normal residual, and Kutta-condition error.

For the 40-panel wrapper test, the constant method returned 40 strengths and a $40\times40$ system. The linear method returned 41 nodal strengths and a $41\times41$ system. The corresponding tangential velocity and $C_p$ arrays each contained 40 values.

### 1.3 Selected post-processing code

```python
# Pressure coefficient at panel control points
Cp = 1.0 - (V_t / V_inf)**2

# Typical linear-strength system and post-processing sequence
A_normal, T_tangent = calculate_velocity_coefficients(panels)
M, b = assemble_linear_system(panels, A_normal, V_inf, alpha)
gamma = solve_vortex_strengths(M, b)
V_t, Cp = tangential_velocity_and_cp(panels, T_tangent, gamma, V_inf, alpha)
```

The snippets are included to show the central computational sequence. They rely on the supporting geometry and influence-integral functions in the implementation notebook.


## 2. Verification on a circular cylinder

### 2.1 Analytical reference

For inviscid, incompressible, irrotational flow around a circular cylinder without circulation, the analytical surface pressure coefficient is

$$
C_{p,\mathrm{exact}}=1-4\sin^2\phi,
$$

where $phi$ is the polar angle around the cylinder. The numerical $C_p$ distribution is compared against this analytical solution.

### 2.2 Linear-strength method: convergence

| Number of panels | RMS \(C_p\) error | Maximum absolute \(C_p\) error | RMS normal residual |
|---:|---:|---:|---:|
| 20 | 0.185898 | 0.296141 | $9.08\times10^{-16}$ |
| 40 | 0.089058 | 0.144537 | $9.63\times10^{-16}$ |
| 80 | 0.043505 | 0.070934 | $1.16\times10^{-15}$ |
| 160 | 0.021490 | 0.035079 | $1.39\times10^{-15}$ |
| 320 | 0.010679 | 0.017436 | $5.80\times10^{-15}$ |

**Discussion.** The RMS error decreases monotonically as the panel count increases. From 20 to 320 panels, the RMS error reduces from approximately 0.186 to 0.0107. The normal residual remains close to machine precision. Thus, the linear-strength implementation satisfies the discrete no-penetration condition, and its surface pressure prediction approaches the analytical cylinder solution as the discretisation is refined.

### 2.3 Constant-strength method: convergence

| Number of panels | RMS \(C_p\) error | Maximum absolute \(C_p\) error |
|---:|---:|---:|
| 21 | $2.52\times10^{-15}$ | $6.22\times10^{-15}$ |
| 41 | $9.81\times10^{-15}$ | $2.40\times10^{-14}$ |
| 81 | $1.90\times10^{-14}$ | $5.15\times10^{-14}$ |
| 161 | $8.40\times10^{-14}$ | $2.05\times10^{-13}$ |
| 321 | $1.27\times10^{-13}$ | $3.13\times10^{-13}$ |

**Discussion.** For the tested odd panel counts, the constant-strength cylinder solution agrees with the analytical pressure coefficient to near machine precision. The circulation and normal-velocity residuals are also of order $10^{-15}$. This result is specific to the cylinder implementation and tested discretisations; it should not be interpreted as proof that the constant-strength airfoil implementation is equally accurate.

At the 40-panel comparison, the RMS difference between the constant- and linear-strength cylinder $C_p$ distributions is approximately 0.08679. The two discretisations therefore need not give identical finite-mesh results, even though each may satisfy its own algebraic constraints.


## 3. Airfoil verification and convergence

The airfoil study considers:

- **NACA 0012:** symmetric airfoil, 12% thickness and zero camber.
- **NACA 4412:** cambered airfoil, 4% maximum camber at 40% chord and 12% thickness.

The convergence calculations use $$V_\infty=1$$ and $$alpha=4^\circ$$

### 3.1 Linear-strength method: NACA 0012

| Panels | \(C_L\) | \(C_{D,p}\) | Normal RMS |
|---:|---:|---:|---:|
| 40 | 0.470687 | 0.010065 | $2.62\times10^{-16}$ |
| 80 | 0.476789 | 0.004672 | $4.72\times10^{-16}$ |
| 160 | 0.479749 | 0.002241 | $8.96\times10^{-16}$ |
| 320 | 0.481201 | 0.001097 | $1.97\times10^{-15}$ |
| 640 | 0.481919 | 0.000542 | $3.54\times10^{-15}$ |

**Discussion.** The lift coefficient approaches a stable value as the number of panels increases. The pressure-drag coefficient decreases toward zero with refinement. This is consistent with the inviscid potential-flow model, which does not represent viscous profile drag. The normal residual remains very small, showing that the linear system satisfies the imposed no-penetration conditions.

### 3.2 Linear-strength method: NACA 4412

| Panels | \(C_L\) | \(C_{D,p}\) |
|---:|---:|---:|
| 40 | 0.503863 | 0.006429 |
| 80 | 0.511440 | 0.002948 |
| 160 | 0.514955 | 0.001407 |
| 320 | 0.516636 | 0.000688 |
| 640 | 0.517457 | 0.000340 |

**Discussion.** The lift coefficient increases gradually toward a limiting value as the mesh is refined. The nonzero lift of the cambered airfoil is physically consistent with its asymmetric geometry. The pressure-drag coefficient decreases with refinement, again tending toward the ideal inviscid result.

### 3.3 Constant-strength airfoil results

| Panels | NACA 0012 \(C_L\) | NACA 0012 \(C_{D,p}\) | NACA 4412 \(C_L\) | NACA 4412 \(C_{D,p}\) |
|---:|---:|---:|---:|---:|
| 40 | 0.502878 | -0.002789 | 0.995799 | -0.057052 |
| 80 | 0.542896 | -0.976455 | 0.951032 | -0.131680 |
| 160 | 0.726712 | -3.555818 | 0.991111 | -0.023424 |
| 320 | 0.964536 | -6.915022 | 0.924411 | -0.199488 |
| 640 | 1.282655 | -11.448805 | 0.846617 | -0.384925 |

**Discussion.** These constant-strength airfoil results do not show a stable convergence trend. The lift values vary substantially with panel count, and the pressure-drag values become large and negative. Therefore, the current constant-strength airfoil implementation should not be considered quantitatively reliable on the basis of this convergence study. This contrasts with the more orderly convergence obtained using the linear-strength method.

**Residual interpretation.** In the constant-strength airfoil system, the last normal-flow equation is replaced by the Kutta condition. The reported maximum normal residual is evaluated at all panel control points, including the one whose normal-flow equation is not part of the solved system. In the 40-panel wrapper test, the maximum residual is approximately $$6.37\times10^{-2}$$. The linear-strength 40-panel test gives approximately $$5.55\times10^{-16}$$. This distinction is important when interpreting the residuals.


## 4. Effect of panel distribution

Uniform and cosine panel spacing were compared at 160 panels and $alpha=4^\circ$. Cosine spacing clusters panels near the leading and trailing edges, while uniform spacing distributes them evenly along the chord.

### 4.1 Linear-strength results

| Airfoil | Distribution | \(C_L\) | \(C_{D,p}\) |
|---|---|---:|---:|
| NACA 0012 | Uniform | 0.473563 | 0.010074 |
| NACA 0012 | Cosine | 0.479749 | 0.002241 |
| NACA 4412 | Uniform | 0.986752 | 0.009436 |
| NACA 4412 | Cosine | 0.993579 | 0.002243 |

**Discussion.** The linear-strength lift values are relatively close between the two distributions in this comparison, but the pressure-drag estimates differ. Cosine spacing provides more geometric resolution near the leading and trailing edges, where curvature and pressure gradients can be strong. The NACA 4412 values in this single distribution comparison should be interpreted cautiously because the uniform-spacing lift differs from the corresponding panel-count convergence sequence. A systematic comparison should use identical geometry conventions and additional refinement levels.

### 4.2 Constant-strength results

| Airfoil | Distribution | \(C_L\) | \(C_{D,p}\) |
|---|---|---:|---:|
| NACA 0012 | Uniform | 2.963786 | 0.398649 |
| NACA 0012 | Cosine | 0.726712 | -3.555818 |
| NACA 4412 | Uniform | 2.077726 | 0.261105 |
| NACA 4412 | Cosine | 0.991111 | -0.023424 |

The constant-strength results vary strongly with panel distribution and include nonphysical pressure-drag values. This reinforces the conclusion that the current constant-strength airfoil implementation requires further numerical validation before its airfoil force predictions can be relied upon.

### 4.3 Physical significance

Panel distribution is not merely a plotting or geometry choice. It changes where the surface is resolved most finely. Clustering panels near high-curvature regions can improve the representation of the geometry and the surface velocity variation. However, panel clustering alone does not guarantee accuracy; the solution must still be assessed through convergence, residuals, and comparison with a trusted reference.


## 5. Prediction of airfoil characteristics and \(C_p\) distributions

The two airfoils are used to demonstrate the predicted surface pressure distributions and force coefficients.

### 5.1 Symmetric airfoil: NACA 0012

At zero angle of attack, the saved 40-panel linear-strength calculation gives:

- $$C_L\approx -8.6\times10^{-16}$$
- upper/lower pressure-coefficient symmetry error approximately $$4.2\times10^{-15}$$
- maximum normal residual approximately $$6.7\times10^{-16}$$

This is an important symmetry check: a symmetric airfoil at zero incidence should have no net lift in the ideal symmetric calculation. At positive incidence $$4^\circ$$, the upper surface accelerates more strongly over much of the chord, producing lower pressure and positive lift. The linear-strength convergence sequence gives $C_L=0.481919$ at 640 panels.

### 5.2 Cambered airfoil: NACA 4412

The NACA 4412 airfoil has positive camber and therefore produces positive lift even at zero incidence. The saved 40-panel linear-strength result at zero incidence gives $C_L\approx0.503863$. At $4^\circ$, the linear-strength convergence sequence approaches $C_L\approx0.517457$ at 640 panels.

### 5.3 Pressure coefficient interpretation

The pressure coefficient is obtained from the local tangential velocity:

$$
C_p=1-\left(\frac{V_t}{V_\infty}\right)^2.
$$

A region of high local speed has a lower (more negative) pressure coefficient. The upper-surface suction peak and the pressure difference between the upper and lower surfaces are responsible for the predicted lift. The notebook's $C_p$-versus-chord plots show these distributions for both airfoils.

The pressure-force integration does not include skin friction. Consequently, $C_{D,p}$ is a pressure-based inviscid force coefficient, not the total viscous profile drag of a real airfoil.


## 6. Effect of increasing airfoil thickness and model limitations

The thickness study uses symmetric NACA 00-series sections at $alpha=4^\circ$, with the linear-strength method. The coarse solution uses 40 panels and is compared with a 320-panel solution. The RMS difference between the two $C_p$ distributions is used as a mesh-sensitivity measure.

### 6.1 Thickness-study results

| Thickness ratio | Airfoil | RMS coarse/fine \(C_p\) difference |
|---:|---|---:|
| 6% | NACA 0006 | 0.154753 |
| 12% | NACA 0012 | 0.117897 |
| 18% | NACA 0018 | 0.088661 |
| 24% | NACA 0024 | 0.076063 |

Additional fine-grid predictions from the same study are:

| Thickness ratio | $C_L$ | $C_{D,p}$ | Maximum $|C_p|$ |
|---:|---:|---:|---:|
| 6%  | 0.457874 | 0.002438 | 3.041072 |
| 12% | 0.479749 | 0.002241 | 1.481841 |
| 18% | 0.501788 | 0.002628 | 1.362380 |
| 24% | 0.524031 | 0.003169 | 1.499216 |

### 6.2 Discussion of the assignment's requested trend

The measured RMS coarse/fine $C_p$ difference decreases as thickness increases, from 0.154753 at 6% thickness to 0.076063 at 24% thickness. Therefore, **the current numerical results do not demonstrate that predictive capability decreases as airfoil thickness increases**.

This RMS difference is a mesh-sensitivity measure, not an absolute prediction error. A coarse and fine solution can agree with each other and still differ from experimental data or a higher-fidelity viscous solution. It would be incorrect to claim that this table proves the requested trend.

### 6.3 Physical limitations of the vortex panel method

The present method models inviscid, incompressible, irrotational flow. It does not model:

- viscous boundary-layer growth and displacement effects;
- skin-friction drag and boundary-layer transition;
- flow separation, separated wakes, and stall;
- viscous pressure recovery;
- compressibility effects and shock waves.

As airfoil thickness and loading increase, real-flow viscous effects and possible separation can become important. The potential-flow panel method cannot represent these phenomena directly. A convincing demonstration of reduced predictive capability with thickness would require comparison against experimental measurements or a validated viscous CFD solution for each thickness at matched Reynolds number and angle of attack.

**Conclusion for this section:** the current study documents how the predicted pressure distribution changes with thickness and mesh refinement, but it does not verify the monotonic loss of prediction capability requested in the assignment.


## 7. Overall conclusions

1. The required constant-strength and linear-strength vortex panel algorithms were implemented, and the required wrapper functions return the calculated strengths, velocities, pressure coefficients, matrices, and residual diagnostics.
2. The linear-strength cylinder solution converges toward the analytical cylinder pressure distribution as the panel count increases from 20 to 320.
3. The constant-strength cylinder implementation gives near-machine-precision agreement for the tested odd panel counts. This cylinder result does not establish accuracy for airfoil calculations.
4. The linear-strength method gives orderly lift convergence for NACA 0012 and NACA 4412. The current constant-strength airfoil results do not show reliable convergence.
5. Panel distribution affects the predicted force coefficients. Cosine spacing concentrates panels near the leading and trailing edges, but convergence checks remain necessary.
6. The symmetric NACA 0012 gives effectively zero lift at zero incidence, while the cambered NACA 4412 gives positive lift even at zero incidence.
7. The thickness study shows a decreasing coarse/fine $C_p$ difference with increasing thickness. It therefore does not establish the requested decrease in predictive capability.
8. The method is useful for idealised potential-flow pressure distributions but cannot predict viscous drag, transition, separation, stall, or shock effects.

## 8. References and acknowledgement

### References

1. Anderson, J. D., *Fundamentals of Aerodynamics*, 6th ed., McGraw-Hill Education.
2. Katz, J. and Plotkin, A., *Low-Speed Aerodynamics*, 2nd ed., Cambridge University Press.
3. AE611 course material and assignment discussions on two-dimensional vortex panel methods.

### Acknowledgement of AI assistance

OpenAI ChatGPT was used as an AI-assisted learning and coding-support tool. Its assistance included explaining the physical basis of constant- and linear-strength vortex panel methods, discussing boundary conditions and the Kutta condition, helping organise selected Python routines and convergence studies, and supporting the interpretation and presentation of numerical results. The student is responsible for checking the implementation, executing the notebook, verifying the numerical results, and ensuring that the submitted work reflects their own understanding.

---

## Final submission checklist

- [ ] Enter student name and student ID.
- [ ] Keep the full implementation in the separate working notebook or required code location.
- [ ] Run the report notebook from top to bottom.
- [ ] Add/export the final \(C_p\), cylinder convergence, airfoil convergence, spacing, and thickness plots from the executed study notebook if the tutor expects figures embedded in the report.
- [ ] Confirm that the numerical tables match the final executed outputs.
- [ ] Add any additional references or colleague discussions actually used.
- [ ] Submit the `.ipynb` report through the required GitHub pull request.


In [ ]:
# Selected example: the two required solver wrappers
# (The full supporting functions are maintained in the implementation notebook.)

result_constant = vortex_panel_method_0(panels, V_inf, alpha)
result_linear = vortex_panel_method_1(panels, V_inf, alpha)

print("Constant-strength:", result_constant["gamma"].shape,
      result_constant["V_t"].shape, result_constant["Cp"].shape)
print("Linear-strength:", result_linear["gamma"].shape,
      result_linear["V_t"].shape, result_linear["Cp"].shape)


### Note on the selected code cell

The final cell is a short usage example, not a self-contained reimplementation. It assumes that the geometry, solver functions, `panels`, `V_inf`, and `alpha` have already been defined by the implementation notebook. The report is intentionally focused on results and discussion rather than duplicating the complete source code.